# UABR — Preflight (version 2): is `data_v2` worth 5 GPU-hours?

**Kaggle settings:** Accelerator **GPU T4 x2** · Internet **On** · **+ Add Input → Your Work → the Phase 2 v2 notebook** (`data_v2`).

A 1/10-scale rehearsal of Phases 3 + 4 on the same code path (~45 min):
1. Pretrain the same 40M model on `data_v2/pretrain_preflight` (~105M tokens, every fact exposure, ~1/10 of the Wikipedia text).
2. Probe recall by exposure tier in **sentence** and **question** form.
3. Phase 4 delegation training in the natural chat format (`question`: decision right after `<ASSISTANT>`).
4. `scripts/preflight_check.py` prints **GO** or **STOP** by rules fixed before any v2 result:
   question recall works · recall rises with exposure · the Phase 4 decision is real (mutual information and tier-sensitive checks pass).

`QUICK = True` → Run All (~10 min, pipeline check only). `QUICK = False` → **Save Version › Save & Run All (Commit)**.

**Re-run without re-pretraining:** attach the earlier preflight notebook's output as a second input (+ Add Input → Your Work). Its finished model is found under `runs/preflight_40m` and reused, so only the probes and Phase 4 run (~15 min).

In [ ]:
QUICK = True
REPO_URL = "https://github.com/GTX-Pradeep/uabr-slm.git"

import os, glob, json, subprocess
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/uabr_slm"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

found = glob.glob("/kaggle/input/**/data_v2/manifest.json", recursive=True)
assert found, "data_v2 not found: Add Input -> Your Work -> the Phase 2 v2 notebook"
DATA = os.path.dirname(found[0])
m = json.load(open(f"{DATA}/manifest.json"))
assert m["data_version"] == "data_v2" and m.get("pretrain_preflight"), "this data has no preflight corpus"
RUN = f"{WORK}/runs/preflight_40m"
# Re-run: if an earlier preflight notebook is attached as an input, reuse its finished preflight model (saves ~30 min)
prev = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/runs/preflight_40m/manifest.json", recursive=True)
        if json.load(open(f)).get("status") == "completed"]
SKIP_PRETRAIN = bool(prev) and not QUICK
if SKIP_PRETRAIN:
    RUN = prev[0]
print("preflight model:", RUN, "(reused from an attached earlier run)" if SKIP_PRETRAIN else "(trained in this run)")
KNOW = f"{WORK}/facts_knowledge_preflight.parquet"
DELEG = f"{WORK}/runs/preflight_delegation" + ("_quick" if QUICK else "")
print("DATA :", DATA)
print(f"preflight corpus: {m['pretrain_preflight']['train_tokens']:,} tokens | full corpus: {m['pretrain']['train_tokens']:,}")
!git log --oneline -1
!python -m pytest tests -q -x

## 1. Preflight pretraining (~30 min; QUICK: 300 steps)

In [ ]:
steps = "--max_steps 300 --save_every 100" if QUICK else ""
if SKIP_PRETRAIN:
    print("skipping pretraining: reusing", RUN)
else:
    !rm -rf {RUN}
    !python src/train/pretrain.py --config configs/pretrain_preflight.yaml --data_dir {DATA}/pretrain_preflight --output_dir {RUN} --compile {steps}

## 2. Recall by exposure tier: sentence (cloze) and question format

In [ ]:
!python scripts/probe_facts.py --ckpt {RUN} --data {DATA} --per_tier 300 --out {WORK}/probe_preflight.json
!python scripts/label_knowledge.py --base {RUN} --data {DATA} --out {KNOW}

## 3. Phase 4 at preflight scale: decision right after `<ASSISTANT>` (format `question`)

In [ ]:
quick = "--quick" if QUICK else ""
!rm -rf {DELEG}
!python scripts/train_delegation.py --base {RUN} --data {DATA} --knowledge {KNOW} --out {DELEG} --format question {quick}

## 3b. Routing-signal diagnostic (no training, ~5 min)
Does the frozen preflight model *internally* separate facts it can answer from facts it cannot? AUROC of output confidence and of linear probes on the hidden state, for "the direct answer is wrong". A high probe AUROC at the decision position means the information is there and the Phase 4 recipe is the bottleneck.

In [ ]:
!python scripts/diagnose_routing.py --model {RUN} --data {DATA} --out {WORK}/routing_diagnostic.json

## 4. GO / STOP

In [ ]:
!python scripts/preflight_check.py --knowledge {KNOW} --gate {DELEG}/gate_report.json --out {WORK}/preflight_report.json

## 5. Next
- **GO** → run Phase 3 on `data_v2` (`notebooks/phase3_pretrain.ipynb`, `DATA_VERSION = "data_v2"`), then Phase 4.
- **STOP** → do not spend the 5 h. Send the output of sections 2–4: the failing check says what to change.